# Step 0 · [1] Candidate pool

Applies the selection rules of `config/sector.yaml` and returns **one row per DOCDB family** with a 0/1 flag per rule:

| Flag | Rule |
|---|---|
| `r_core_cpc`, `r_core_ipc`, `r_keyword` | **A** · core codes or specific keywords |
| `r_component_cpc`, `r_component_ipc` | **B** · generic codes **and** an eyewear term in the family |
| `r_eyewear_cpc`, `r_eyewear_ipc` | **C** · spectacle codes **and** an electronic term in the family |
| `r_review` | head-up / head-mounted display codes |
| `r_excluded` | an exclusion term appears in the family |

Two tiers (v0.3, after eyeballing the PROD samples: rules B and C alone were too noisy):

- `in_sector` = rule **A**. Exclusion terms are not applied here: a core patent that mentions a windshield once is still a smart-glasses patent.
- `to_review` = rule **B**, **C** or `r_review`, **not** A, and not excluded: these go to the semantic filter (step 3).

Rules as in the tutorials: DOCDB families, `ipr_type = 'PI'`, no dummy applications, earliest priority year in the window. Everything is computed in the database; only the flags are downloaded.

Run on **TEST** first, then **PROD**.

In [ ]:
%pip install -q pyyaml

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from epo.tipdata.patstat import PatstatClient

from src.sector import CORE_RULES, REVIEW_RULES, candidate_pool_query, family_titles, load_config

ENV = "TEST"                       # "TEST" to check the code, "PROD" for the real numbers
OUTPUT_DIR = Path("outputs/step0")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_colwidth", 100)

cfg = load_config()
patstat = PatstatClient(env=ENV)
db = patstat.orm()

YEAR_MIN, YEAR_MAX = cfg["years"]["min"], cfg["years"]["max"]
LAST_COMPLETE_YEAR = YEAR_MAX - cfg["years"]["truncated"]
print(f"{cfg['sector_name']} | env {ENV} | years {YEAR_MIN}-{YEAR_MAX} | complete up to {LAST_COMPLETE_YEAR}")

## 1. The candidate pool

In [ ]:
cand = patstat.df(candidate_pool_query(db, cfg))

rule_cols = [c for c in cand.columns if c.startswith("r_")]
cand[rule_cols] = cand[rule_cols].astype(int)
core = cand[list(CORE_RULES)].max(axis=1) == 1
review = cand[list(REVIEW_RULES)].max(axis=1) == 1
cand["in_sector"] = core
cand["to_review"] = ~core & review & (cand["r_excluded"] == 0)
cand["truncated"] = cand["year"] > LAST_COMPLETE_YEAR

pd.Series({
    "candidate families": len(cand),
    "in sector (rule A)": int(cand["in_sector"].sum()),
    "to review (B, C or HUD/HMD only)": int(cand["to_review"].sum()),
    "  of which r_review only": int((cand["to_review"] & (cand[list(REVIEW_RULES[:-1])].max(axis=1) == 0)).sum()),
    "dropped by exclude_terms": int((~core & review & (cand["r_excluded"] == 1)).sum()),
}, name="families").to_frame()

## 2. Contribution of each rule

As in Tutorial 0, Section 5.6: for each rule, how many candidate families it finds, how many **only** that rule finds, and where they end up.

In [ ]:
all_rules = list(CORE_RULES + REVIEW_RULES)
n_rules = cand[all_rules].sum(axis=1)
contribution = pd.DataFrame({
    "families": cand[all_rules].sum(),
    "only this rule": cand[n_rules == 1][all_rules].sum(),
    "in sector": cand[cand["in_sector"]][all_rules].sum(),
    "to review": cand[cand["to_review"]][all_rules].sum(),
})
contribution

## 3. Families per priority year

In [ ]:
per_year = (
    pd.concat([cand[cand["in_sector"]].assign(group="in sector (rule A)"),
               cand[cand["to_review"]].assign(group="to review")])
    .groupby(["year", "group"]).size().unstack(fill_value=0)
)

fig, ax = plt.subplots(figsize=(10, 4.5))
per_year.plot(ax=ax, marker="o", ms=3)
ax.axvspan(LAST_COMPLETE_YEAR + 0.5, YEAR_MAX + 0.5, color="grey", alpha=0.15, lw=0, label="incomplete years")
ax.set(title=f"{cfg['sector_name']}: candidate families per earliest priority year ({ENV})",
       xlabel="earliest priority year", ylabel="families", xlim=(YEAR_MIN, YEAR_MAX))
ax.legend()
plt.show()

## 4. Eyeballing: random titles

For each rule, families found **only** by that rule (the riskiest ones), plus excluded families. Read them: are they smart glasses?

In [ ]:
N_SAMPLE = 8


def one_title_per_family(titles):
    """Prefer an English title; otherwise any title, prefixed with its language."""
    lang = titles["appln_title_lg"].str.strip().str.lower()       # PATSTAT may store "EN" or "en"
    titles = titles.assign(is_en=lang == "en").sort_values("is_en", ascending=False)
    first = titles.drop_duplicates("docdb_family_id")
    return [t if en else f"[{lg}] {t}" for en, lg, t in zip(first["is_en"], first["appln_title_lg"], first["appln_title"])]


def show_sample(label, families):
    ids = families["docdb_family_id"].sample(min(N_SAMPLE, len(families)), random_state=42)
    if ids.empty:
        print(f"--- {label}: none")
        return
    titles = patstat.df(family_titles(db, ids))
    print(f"--- {label} ({len(families):,} families)")
    if titles.empty:
        print("    (no titles in PATSTAT for these families)")
        return
    for t in one_title_per_family(titles):
        print("   ", t)


only_one = cand[n_rules == 1]
for rule in all_rules:
    show_sample(f"only {rule}", only_one[only_one[rule] == 1])
show_sample("dropped by exclude_terms", cand[~cand["in_sector"] & ~cand["to_review"] & (cand["r_excluded"] == 1)])

## 5. Save

The table stays on TIP (`outputs/` is git-ignored): the repository keeps the config and the queries that rebuild it.

In [ ]:
path = OUTPUT_DIR / f"candidates_{ENV}.parquet"
try:
    cand.to_parquet(path, index=False)
except ImportError:                       # no pyarrow: fall back to CSV
    path = path.with_suffix(".csv")
    cand.to_csv(path, index=False)
print(f"saved {len(cand):,} families to {path}")